In [1]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

In [37]:
import numpy as np

def print_plain_english_report(y_true, y_pred):
    # Convert inputs to numpy arrays to make filtering easier
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    total = len(y_true)
    
    # --- Class 0: Healthy Patients ---
    actual_healthy = np.sum(y_true == 0)
    pred_healthy = np.sum(y_pred == 0)
    correct_healthy = np.sum((y_true == 0) & (y_pred == 0))
    
    catch_rate_healthy = (correct_healthy / actual_healthy) * 100
    guess_accuracy_healthy = (correct_healthy / pred_healthy * 100) if pred_healthy > 0 else 0.0
    
    # --- Class 1: Heart Disease Patients ---
    actual_sick = np.sum(y_true == 1)
    pred_sick = np.sum(y_pred == 1)
    correct_sick = np.sum((y_true == 1) & (y_pred == 1))
    
    catch_rate_sick = (correct_sick / actual_sick) * 100
    guess_accuracy_sick = (correct_sick / pred_sick * 100) if pred_sick > 0 else 0.0
    
    # --- Overall Score ---
    overall_correct = np.sum(y_true == y_pred)
    overall_accuracy = (overall_correct / total) * 100
    
    # --- Print the Beautiful human-readable report! ---
    print("-" * 60)
    print("🩺 HEART DISEASE CLASSIFIER - HUMAN ENGLISH REPORT")
    print("-" * 60)
    print(f"Total Patients Tested: {total}")
    print()
    print("HEALTHY PATIENTS (Class 0):")
    print(f"- Total healthy patients tested: {actual_healthy}")
    print(f"- The model correctly diagnosed {correct_healthy} as healthy.")
    print(f"- Catch Rate: The model successfully found {catch_rate_healthy:.1f}% of all healthy patients.")
    print(f"- Guess Accuracy: When the model predicted healthy, it was correct {guess_accuracy_healthy:.1f}% of the time.")
    print()
    print("HEART DISEASE PATIENTS (Class 1):")
    print(f"- Total sick patients tested: {actual_sick}")
    print(f"- The model correctly diagnosed {correct_sick} as having heart disease.")
    print(f"- Catch Rate: The model successfully found {catch_rate_sick:.1f}% of all sick patients.")
    print(f"- Guess Accuracy: When the model predicted heart disease, it was correct {guess_accuracy_sick:.1f}% of the time.")
    print()
    print("OVERALL PERFORMANCE:")
    print(f"- The model got {overall_accuracy:.1f}% of all its predictions correct.")
    print("-" * 60)

In [2]:
df = pd.read_csv('heart_disease.csv')
df


,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,...,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,Yes,Yes,No,24.991591,Yes,...,No,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,No
1,69.0,Female,146.0,286.0,High,No,Yes,Yes,25.221799,No,...,No,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,No
2,46.0,Male,126.0,216.0,Low,No,No,No,29.855447,No,...,Yes,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,No
3,32.0,Female,122.0,293.0,High,Yes,Yes,No,24.130477,Yes,...,Yes,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,No
4,60.0,Male,166.0,242.0,Low,Yes,Yes,Yes,20.486289,Yes,...,No,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,25.0,Female,136.0,243.0,Medium,Yes,No,No,18.788791,Yes,...,Yes,Medium,High,6.834954,Medium,343.0,133.0,3.588814,19.132004,Yes
9996,38.0,Male,172.0,154.0,Medium,No,No,No,31.856801,Yes,...,Yes,NaN,High,8.247784,Low,377.0,83.0,2.658267,9.715709,Yes
9997,73.0,Male,152.0,201.0,High,Yes,No,Yes,26.899911,No,...,Yes,NaN,Low,4.436762,Low,248.0,88.0,4.408867,9.492429,Yes
9998,23.0,Male,142.0,299.0,Low,Yes,No,Yes,34.964026,Yes,...,Yes,Medium,High,8.526329,Medium,113.0,153.0,7.215634,11.873486,Yes


In [3]:
df.columns

Index(['Age', 'Gender', 'Blood Pressure', 'Cholesterol Level',
       'Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes', 'BMI',
       'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol',
       'Alcohol Consumption', 'Stress Level', 'Sleep Hours',
       'Sugar Consumption', 'Triglyceride Level', 'Fasting Blood Sugar',
       'CRP Level', 'Homocysteine Level', 'Heart Disease Status'],
      dtype='str')

In [4]:
df.isnull().sum()

Age                       29
Gender                    19
Blood Pressure            19
Cholesterol Level         30
Exercise Habits           25
Smoking                   25
Family Heart Disease      21
Diabetes                  30
BMI                       22
High Blood Pressure       26
Low HDL Cholesterol       25
High LDL Cholesterol      26
Alcohol Consumption     2586
Stress Level              22
Sleep Hours               25
Sugar Consumption         30
Triglyceride Level        26
Fasting Blood Sugar       22
CRP Level                 26
Homocysteine Level        20
Heart Disease Status       0
dtype: int64

In [41]:
df = df.drop([
    'Exercise Habits',
    'Alcohol Consumption',
    'Stress Level',
    'Sleep Hours',
    'Sugar Consumption'
], axis=1, errors='ignore')

In [42]:
df

,Age,Blood Pressure,Cholesterol Level,BMI,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Gender_Male,Smoking_Yes,Family Heart Disease_Yes,Diabetes_Yes,High Blood Pressure_Yes,Low HDL Cholesterol_Yes,High LDL Cholesterol_Yes,Heart Disease Status_Yes
0,56.0,153.0,155.0,24.991591,342.0,120.0,12.969246,12.387250,1,1,1,0,1,1,0,0
1,69.0,146.0,286.0,25.221799,133.0,157.0,9.355389,19.298875,0,0,1,1,0,1,0,0
2,46.0,126.0,216.0,29.855447,393.0,92.0,12.709873,11.230926,1,0,0,0,0,1,1,0
3,32.0,122.0,293.0,24.130477,293.0,94.0,12.509046,5.961958,0,1,1,0,1,0,1,0
4,60.0,166.0,242.0,20.486289,263.0,154.0,10.381259,8.153887,1,1,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,25.0,136.0,243.0,18.788791,343.0,133.0,3.588814,19.132004,0,1,0,0,1,0,1,1
9996,38.0,172.0,154.0,31.856801,377.0,83.0,2.658267,9.715709,1,0,0,0,1,0,1,1
9997,73.0,152.0,201.0,26.899911,248.0,88.0,4.408867,9.492429,1,1,0,1,0,1,1,1
9998,23.0,142.0,299.0,34.964026,113.0,153.0,7.215634,11.873486,1,1,0,1,1,0,1,1


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       10000 non-null  float64
 1   Blood Pressure            10000 non-null  float64
 2   Cholesterol Level         10000 non-null  float64
 3   BMI                       10000 non-null  float64
 4   Triglyceride Level        10000 non-null  float64
 5   Fasting Blood Sugar       10000 non-null  float64
 6   CRP Level                 10000 non-null  float64
 7   Homocysteine Level        10000 non-null  float64
 8   Gender_Male               10000 non-null  int64  
 9   Smoking_Yes               10000 non-null  int64  
 10  Family Heart Disease_Yes  10000 non-null  int64  
 11  Diabetes_Yes              10000 non-null  int64  
 12  High Blood Pressure_Yes   10000 non-null  int64  
 13  Low HDL Cholesterol_Yes   10000 non-null  int64  
 14  High LDL Cholester

In [40]:
df.isnull().sum()

Age                         0
Blood Pressure              0
Cholesterol Level           0
BMI                         0
Triglyceride Level          0
Fasting Blood Sugar         0
CRP Level                   0
Homocysteine Level          0
Gender_Male                 0
Smoking_Yes                 0
Family Heart Disease_Yes    0
Diabetes_Yes                0
High Blood Pressure_Yes     0
Low HDL Cholesterol_Yes     0
High LDL Cholesterol_Yes    0
Heart Disease Status_Yes    0
dtype: int64

In [22]:
# Get all numeric columns
# select_dtypes() is used to select columns based on data types
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns #choose integer and float format
print(numeric_columns)

# Get all categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns #object is text format
print(categorical_columns)


# Fill NaN values in numeric columns using median
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median()) 

# Fill NaN values in categorical columns using mode
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

Index(['Age', 'Blood Pressure', 'Cholesterol Level', 'BMI',
       'Triglyceride Level', 'Fasting Blood Sugar', 'CRP Level',
       'Homocysteine Level', 'Gender_Male', 'Smoking_Yes',
       'Family Heart Disease_Yes', 'Diabetes_Yes', 'High Blood Pressure_Yes',
       'Low HDL Cholesterol_Yes', 'High LDL Cholesterol_Yes',
       'Heart Disease Status_Yes'],
      dtype='str')
Index([], dtype='str')


In [24]:
df.isnull().sum()

Age                         0
Blood Pressure              0
Cholesterol Level           0
BMI                         0
Triglyceride Level          0
Fasting Blood Sugar         0
CRP Level                   0
Homocysteine Level          0
Gender_Male                 0
Smoking_Yes                 0
Family Heart Disease_Yes    0
Diabetes_Yes                0
High Blood Pressure_Yes     0
Low HDL Cholesterol_Yes     0
High LDL Cholesterol_Yes    0
Heart Disease Status_Yes    0
dtype: int64

In [25]:
## Convert categorical variables into numeric variables(True and False)
df = pd.get_dummies(df, drop_first=True)

# Convert boolean columns into integers
bool_columns = df.select_dtypes(include=['bool']).columns

for col in bool_columns:
    df[col] = df[col].astype(int) 
df

,Age,Blood Pressure,Cholesterol Level,BMI,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Gender_Male,Smoking_Yes,Family Heart Disease_Yes,Diabetes_Yes,High Blood Pressure_Yes,Low HDL Cholesterol_Yes,High LDL Cholesterol_Yes,Heart Disease Status_Yes
0,56.0,153.0,155.0,24.991591,342.0,120.0,12.969246,12.387250,1,1,1,0,1,1,0,0
1,69.0,146.0,286.0,25.221799,133.0,157.0,9.355389,19.298875,0,0,1,1,0,1,0,0
2,46.0,126.0,216.0,29.855447,393.0,92.0,12.709873,11.230926,1,0,0,0,0,1,1,0
3,32.0,122.0,293.0,24.130477,293.0,94.0,12.509046,5.961958,0,1,1,0,1,0,1,0
4,60.0,166.0,242.0,20.486289,263.0,154.0,10.381259,8.153887,1,1,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,25.0,136.0,243.0,18.788791,343.0,133.0,3.588814,19.132004,0,1,0,0,1,0,1,1
9996,38.0,172.0,154.0,31.856801,377.0,83.0,2.658267,9.715709,1,0,0,0,1,0,1,1
9997,73.0,152.0,201.0,26.899911,248.0,88.0,4.408867,9.492429,1,1,0,1,0,1,1,1
9998,23.0,142.0,299.0,34.964026,113.0,153.0,7.215634,11.873486,1,1,0,1,1,0,1,1


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target variable
X = df.drop('Heart Disease Status_Yes', axis=1)
y = df['Heart Disease Status_Yes']

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Create StandardScaler object
scaler = StandardScaler()

# Fit and transform training data
X_train = scaler.fit_transform(X_train)

# Transform testing data
X_test = scaler.transform(X_test)

In [28]:
print(X_train.shape,
      X_test.shape,
      y_train.shape,
      y_test.shape)


(8000, 15) (2000, 15) (8000,) (2000,)


In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize the Random Forest Classifier
# We set random_state=42 for reproducibility (so we always get the same trees)
rf_clf = RandomForestClassifier(random_state=42, class_weight='balanced')

# 2. Train the model (fit it to the training data)
rf_clf.fit(X_train, y_train)

# 3. Predict the labels for our testing features
y_pred = rf_clf.predict(X_test)

# 4. Print the baseline accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Baseline Accuracy: {accuracy:.4f}\n")

# 5. Print the full classification report (for your professor)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# 6. Print the human-readable translation (for you!)
print("\n")
print_plain_english_report(y_test, y_pred)

Baseline Accuracy: 0.8065

Classification Report:
              precision    recall  f1-score   support

           0       0.81      1.00      0.89      1613
           1       0.00      0.00      0.00       387

    accuracy                           0.81      2000
   macro avg       0.40      0.50      0.45      2000
weighted avg       0.65      0.81      0.72      2000



------------------------------------------------------------
🩺 HEART DISEASE CLASSIFIER - HUMAN ENGLISH REPORT
------------------------------------------------------------
Total Patients Tested: 2000

HEALTHY PATIENTS (Class 0):
- Total healthy patients tested: 1613
- The model correctly diagnosed 1613 as healthy.
- Catch Rate: The model successfully found 100.0% of all healthy patients.
- Guess Accuracy: When the model predicted healthy, it was correct 80.7% of the time.

HEART DISEASE PATIENTS (Class 1):
- Total sick patients tested: 387
- The model correctly diagnosed 0 as having heart disease.
- Catch Rate: Th

C:\Users\harri\gb_stuff\school_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\harri\gb_stuff\school_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\harri\gb_stuff\school_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [35]:
from sklearn.model_selection import GridSearchCV

# Initialize the Random Forest
rf_clf_grid = RandomForestClassifier(random_state=42)

# Define the knobs we want to test
param_grid = {
    'n_estimators': [50, 100],          
    'max_depth': [5, 10],               
    'class_weight': ['balanced']        
}

# Set up the Grid Search
grid_search = GridSearchCV(
    estimator=rf_clf_grid, 
    param_grid=param_grid, 
    scoring='f1', 
    cv=3, 
    n_jobs=-1
)

# Run the search
grid_search.fit(X_train, y_train)

# Print the best knobs
print(f"Best Knobs Found: {grid_search.best_params_}\n")

# Make predictions using our best model
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

# ----------------------------------------------------------------
# 1. STANDARD MACHINE LEARNING OUTPUT (FOR THE PROFESSOR)
# ----------------------------------------------------------------
print("================================================================")
print("1. STANDARD MACHINE LEARNING EVALUATION (FOR THE PROFESSOR)")
print("================================================================")
print("Tuned Classification Report:")
print(classification_report(y_test, y_pred_tuned))

# ----------------------------------------------------------------
# 2. PLAIN ENGLISH TRANSLATION (FOR OUR WRITING & CLARITY)
# ----------------------------------------------------------------
print("\n================================================================")
print("2. HUMAN-READABLE TRANSLATION (OUR PLAIN ENGLISH EXPLANATION)")
print("================================================================")
print_plain_english_report(y_test, y_pred_tuned)


Best Knobs Found: {'class_weight': 'balanced', 'max_depth': 5, 'n_estimators': 50}

1. STANDARD MACHINE LEARNING EVALUATION (FOR THE PROFESSOR)
Tuned Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.62      0.70      1613
           1       0.18      0.36      0.24       387

    accuracy                           0.57      2000
   macro avg       0.49      0.49      0.47      2000
weighted avg       0.68      0.57      0.61      2000


2. HUMAN-READABLE TRANSLATION (OUR PLAIN ENGLISH EXPLANATION)
------------------------------------------------------------
🩺 HEART DISEASE CLASSIFIER - HUMAN ENGLISH REPORT
------------------------------------------------------------
Total Patients Tested: 2000

HEALTHY PATIENTS (Class 0):
- Total healthy patients tested: 1613
- The model correctly diagnosed 996 as healthy.
- Catch Rate: The model successfully found 61.7% of all healthy patients.
- Guess Accuracy: When the model predicted healt